# Paper #64 — Flare-Productive Active Regions: Implementation / 구현

**Paper**: Toriumi, S., & Wang, H., "Flare-productive active regions," *Living Reviews in Solar Physics*, 16:3, 2019. DOI: 10.1007/s41116-019-0019-7

This notebook implements the key observational diagnostics and forecasting concepts from the review:
1. Synthetic δ-sunspot magnetogram (opposing polarities in one penumbra)
2. Free magnetic energy: NLFFF vs. potential field difference (toy 2D analog)
3. Flare productivity vs. total unsigned flux (statistical scaling)
4. Shear angle measurement along the polarity inversion line
5. Hale classification decision tree (α → β → βγ → βγδ)
6. Flare forecasting probability by AR class

이 노트북은 리뷰 논문의 핵심 관측 진단과 예보 개념을 구현한다:
1. δ-sunspot 합성 magnetogram
2. 자유 자기 에너지 (NLFFF vs potential 장난감 2D 아날로그)
3. 총 비부호화 자속 대 플레어 생산성 스케일링
4. PIL을 따라 shear angle 측정
5. Hale 분류 결정 트리
6. AR 분류에 따른 플레어 예보 확률

In [ ]:
# Standard imports and plotting configuration.
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

rng = np.random.default_rng(seed=42)

## Part 1: Synthetic δ-Sunspot Magnetogram / δ-흑점 합성 자력계 영상

A δ-sunspot is defined by **umbrae of opposite polarity (<2° apart) sharing a common penumbra** (Künzel 1960). This geometry concentrates the polarity inversion line (PIL) inside a single spot, forcing strong-gradient, highly-sheared transverse fields along the PIL.

δ-sunspot은 공통 penumbra 내 반대 극성의 umbra(<2° 간격)로 정의되며(Künzel 1960), 단일 흑점 안에 PIL이 놓이므로 강경사·강전단 횡방위 자장이 PIL을 따라 집중된다.

In [ ]:
def delta_sunspot_magnetogram(
    n: int = 256,
    extent_arcsec: float = 40.0,
    positive_centre: tuple = (-4.0, 0.0),
    negative_centre: tuple = (4.0, 0.0),
    umbra_radius: float = 3.0,
    common_penumbra_radius: float = 10.0,
    peak_B: float = 3000.0,
) -> tuple:
    """Generate a synthetic line-of-sight magnetogram of a delta-sunspot.

    Args:
        n: Grid resolution (n x n pixels).
        extent_arcsec: Half-width of the field of view in arcseconds.
        positive_centre: (x, y) of positive umbra centre in arcseconds.
        negative_centre: (x, y) of negative umbra centre in arcseconds.
        umbra_radius: Gaussian umbra sigma in arcseconds.
        common_penumbra_radius: Shared penumbra sigma in arcseconds.
        peak_B: Peak umbral field strength in Gauss.

    Returns:
        Tuple (X, Y, Bz, penumbra_mask) with 2-D arcsecond grids and arrays.
    """
    x = np.linspace(-extent_arcsec, extent_arcsec, n)
    y = np.linspace(-extent_arcsec, extent_arcsec, n)
    X, Y = np.meshgrid(x, y)

    r_pos = np.hypot(X - positive_centre[0], Y - positive_centre[1])
    r_neg = np.hypot(X - negative_centre[0], Y - negative_centre[1])
    Bz = peak_B * np.exp(-(r_pos / umbra_radius) ** 2)
    Bz -= peak_B * np.exp(-(r_neg / umbra_radius) ** 2)

    midpoint = (
        0.5 * (positive_centre[0] + negative_centre[0]),
        0.5 * (positive_centre[1] + negative_centre[1]),
    )
    r_pen = np.hypot(X - midpoint[0], Y - midpoint[1])
    penumbra_mask = np.exp(-(r_pen / common_penumbra_radius) ** 2)

    return X, Y, Bz, penumbra_mask


X, Y, Bz, penumbra = delta_sunspot_magnetogram()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(
    Bz,
    origin="lower",
    extent=[X.min(), X.max(), Y.min(), Y.max()],
    cmap="gray_r",
    vmin=-3000,
    vmax=3000,
)
ax.contour(X, Y, penumbra, levels=[0.3], colors="orange", linewidths=2)
ax.contour(X, Y, Bz, levels=[0], colors="cyan", linewidths=2)
ax.set_xlabel("X [arcsec]")
ax.set_ylabel("Y [arcsec]")
ax.set_title("Synthetic δ-sunspot magnetogram / δ-흑점 합성\n"
             "cyan = PIL (Bz=0), orange = common penumbra outline")
plt.colorbar(im, ax=ax, label="B_z [G]")
plt.tight_layout()
plt.show()

## Part 2: Free Magnetic Energy — NLFFF vs. Potential Field / 자유 자기 에너지

The free magnetic energy is the excess energy over the potential (current-free) configuration:

$$E_\mathrm{free} = \frac{1}{8\pi}\int_V \left(B^2_\mathrm{NLFFF} - B^2_\mathrm{Pot}\right) dV.$$

For a flare-productive AR, $E_\mathrm{free}\sim 10^{31}$–$10^{32}$ erg. Below we build a toy 2-D analog: the potential field has no free energy by construction; the non-potential ("NLFFF") field adds a sheared transverse component concentrated on the PIL.

자유 자기 에너지는 potential(current-free) 배치 대비 초과 에너지이다. 플레어 AR은 $10^{31-32}$ erg 수준이며, 여기서는 PIL에 집중된 sheared 횡방위 성분을 더한 2D 장난감 모델로 계산한다.

In [ ]:
def potential_vs_nlfff_transverse(
    Bz: np.ndarray, shear_amplitude: float = 1500.0
) -> tuple:
    """Construct toy 2-D potential and NLFFF transverse field components.

    The potential transverse field is approximated as the gradient of a
    smoothed B_z map (i.e., curl-free and aligned with the normal gradient).
    The NLFFF transverse field is the potential one plus a shear component
    that runs along the PIL (orthogonal to grad B_z).

    Args:
        Bz: 2-D line-of-sight magnetogram [Gauss].
        shear_amplitude: Peak shear-field strength at the PIL [Gauss].

    Returns:
        Tuple (Bx_pot, By_pot, Bx_nl, By_nl) of 2-D arrays [Gauss].
    """
    Bz_s = gaussian_filter(Bz, sigma=4)
    gy, gx = np.gradient(Bz_s)
    norm = np.hypot(gx, gy) + 1e-6
    Bx_pot = 0.4 * gx
    By_pot = 0.4 * gy

    pil_weight = np.exp(-(Bz_s / 600.0) ** 2)
    Bx_shear = -shear_amplitude * pil_weight * gy / norm
    By_shear = shear_amplitude * pil_weight * gx / norm

    return Bx_pot, By_pot, Bx_pot + Bx_shear, By_pot + By_shear


Bx_p, By_p, Bx_n, By_n = potential_vs_nlfff_transverse(Bz)

B2_nl = Bx_n ** 2 + By_n ** 2 + Bz ** 2
B2_pot = Bx_p ** 2 + By_p ** 2 + Bz ** 2
rho_free = (B2_nl - B2_pot) / (8 * np.pi)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(Bz, origin="lower", cmap="gray_r", vmin=-3000, vmax=3000,
               extent=[X.min(), X.max(), Y.min(), Y.max()])
skip = 12
axes[0].quiver(X[::skip, ::skip], Y[::skip, ::skip],
               Bx_n[::skip, ::skip], By_n[::skip, ::skip],
               color="red", scale=2.5e4, label="NLFFF transverse")
axes[0].quiver(X[::skip, ::skip], Y[::skip, ::skip],
               Bx_p[::skip, ::skip], By_p[::skip, ::skip],
               color="blue", scale=2.5e4, label="Potential transverse", alpha=0.6)
axes[0].contour(X, Y, Bz, levels=[0], colors="cyan")
axes[0].set_title("NLFFF (red) vs. Potential (blue) transverse field")
axes[0].set_xlabel("X [arcsec]"); axes[0].set_ylabel("Y [arcsec]")
axes[0].legend(loc="upper right")

im = axes[1].imshow(rho_free, origin="lower", cmap="hot",
                    extent=[X.min(), X.max(), Y.min(), Y.max()])
axes[1].contour(X, Y, Bz, levels=[0], colors="cyan")
axes[1].set_title("Free-energy density ρ_free [erg cm⁻³]")
axes[1].set_xlabel("X [arcsec]"); axes[1].set_ylabel("Y [arcsec]")
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

pixel_arcsec = (X.max() - X.min()) / X.shape[1]
pixel_cm = pixel_arcsec * 7.25e7
dx = dy = pixel_cm
depth_cm = 1e9  # 10 Mm
E_free = rho_free.sum() * dx * dy * depth_cm
print(f"Toy integrated free magnetic energy: {E_free:.2e} erg")
print("Observed X-class release range: 1e31 – 1e32 erg")

## Part 3: Flare Productivity vs. Total Unsigned Flux / 플레어 생산성 대 총 자속

Following Sammis et al. (2000) and related statistical works, larger total unsigned flux generally correlates with higher flare magnitude, but the relation is noisy and complexity (Mount-Wilson class) dominates at the high end. Here we simulate an AR population and reproduce a qualitative version of the Sammis Fig. 10 pattern.

Sammis(2000)에 따르면 총 비부호 자속이 클수록 플레어 강도가 커지는 경향이 있지만, 복잡성(Mount-Wilson class)이 상한을 결정한다. 여기서는 AR 모집단을 시뮬레이션하고 Sammis Fig. 10 패턴을 정성적으로 재현한다.

In [ ]:
def simulate_ar_population(n: int = 400) -> dict:
    """Simulate active-region population with flux and Hale-class labels.

    Args:
        n: Number of synthetic active regions.

    Returns:
        Dict with keys 'flux' (Mx), 'hale_class' (str), 'peak_goes' (W m^-2).
    """
    log_flux = rng.uniform(21.0, 23.2, n)
    flux = 10.0 ** log_flux

    classes = []
    for f in flux:
        if f < 3e21:
            classes.append(rng.choice(["alpha", "beta"], p=[0.7, 0.3]))
        elif f < 1e22:
            classes.append(rng.choice(["beta", "beta_gamma"], p=[0.7, 0.3]))
        elif f < 3e22:
            classes.append(rng.choice(
                ["beta", "beta_gamma", "beta_gamma_delta"],
                p=[0.3, 0.4, 0.3]))
        else:
            classes.append(rng.choice(
                ["beta_gamma", "beta_gamma_delta"], p=[0.3, 0.7]))
    classes = np.array(classes)

    complexity_boost = {
        "alpha": 0.0,
        "beta": 0.3,
        "beta_gamma": 0.7,
        "beta_gamma_delta": 1.2,
    }
    boost = np.array([complexity_boost[c] for c in classes])
    log_goes = -7.5 + 0.8 * (log_flux - 21.0) + boost + rng.normal(0, 0.4, n)
    peak_goes = 10.0 ** log_goes

    return {"flux": flux, "hale_class": classes, "peak_goes": peak_goes}


pop = simulate_ar_population()

fig, ax = plt.subplots(figsize=(9, 6))
class_colours = {
    "alpha": "tab:green",
    "beta": "tab:blue",
    "beta_gamma": "tab:orange",
    "beta_gamma_delta": "tab:red",
}
for cls, colour in class_colours.items():
    mask = pop["hale_class"] == cls
    ax.scatter(pop["flux"][mask], pop["peak_goes"][mask], c=colour,
               s=30, alpha=0.7, label=cls.replace("_", "-"))
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total unsigned flux [Mx]")
ax.set_ylabel("Peak GOES soft X-ray flux [W m⁻²]")
ax.set_title("Flare magnitude vs. AR flux (synthetic population) — Sammis-style")
for y, cls in zip([1e-4, 1e-5, 1e-6, 1e-7], ["X", "M", "C", "B"]):
    ax.axhline(y, color="k", lw=0.5, ls="--")
    ax.text(1.2e21, y * 1.15, cls, fontsize=9)
ax.legend(title="Hale class")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Shear Angle Along the PIL / PIL을 따른 Shear Angle

The shear angle is the angle between the observed transverse field and the potential reference field along the PIL. Flaring PILs show 80°–90° (Hagyard 1990). We measure it on our toy NLFFF/potential fields constructed in Part 2.

Shear angle은 PIL을 따라 관측된 횡방위 자장과 potential 기준 자장 사이의 각도이며, 플레어 PIL에서 80–90°를 보인다(Hagyard 1990).

In [ ]:
def compute_shear_angle(
    Bx_pot: np.ndarray,
    By_pot: np.ndarray,
    Bx_nl: np.ndarray,
    By_nl: np.ndarray,
) -> np.ndarray:
    """Compute the shear angle between NLFFF and potential transverse fields.

    Args:
        Bx_pot: Potential horizontal x-component.
        By_pot: Potential horizontal y-component.
        Bx_nl: NLFFF horizontal x-component.
        By_nl: NLFFF horizontal y-component.

    Returns:
        2-D array of shear angle in degrees (0 – 180).
    """
    dot = Bx_pot * Bx_nl + By_pot * By_nl
    norm_p = np.hypot(Bx_pot, By_pot) + 1e-6
    norm_n = np.hypot(Bx_nl, By_nl) + 1e-6
    cos_theta = np.clip(dot / (norm_p * norm_n), -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))


shear = compute_shear_angle(Bx_p, By_p, Bx_n, By_n)
pil_mask = np.abs(Bz) < 100
pil_shear = shear[pil_mask]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(shear, origin="lower", cmap="plasma",
                    extent=[X.min(), X.max(), Y.min(), Y.max()],
                    vmin=0, vmax=90)
axes[0].contour(X, Y, Bz, levels=[0], colors="cyan", linewidths=2)
axes[0].set_title("Shear angle map [deg]")
axes[0].set_xlabel("X [arcsec]"); axes[0].set_ylabel("Y [arcsec]")
plt.colorbar(im, ax=axes[0])

axes[1].hist(pil_shear, bins=30, color="tab:purple", edgecolor="k")
axes[1].axvline(np.median(pil_shear), color="red", ls="--",
                label=f"Median = {np.median(pil_shear):.1f}°")
axes[1].axvspan(80, 90, color="gold", alpha=0.3, label="Flaring PIL 80–90°")
axes[1].set_xlabel("Shear angle [deg]")
axes[1].set_ylabel("Pixel count")
axes[1].set_title("Distribution along PIL")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Median shear along PIL: {np.median(pil_shear):.1f}° "
      f"(flaring PILs: 80–90°; Hagyard 1990)")

## Part 5: Hale Classification Decision Tree / Hale 분류 결정 트리

We implement a rule-based classifier that reproduces the observational decision process: unipolar → α; balanced bipolar without δ → β; complex without δ → βγ; opposing polarities within common penumbra → βγδ.

관측적 결정 과정을 그대로 구현한다: 단극 → α, 균형 쌍극(δ 없음) → β, 복잡(δ 없음) → βγ, 공통 penumbra 내 반대 극성 → βγδ.

In [ ]:
def classify_hale(
    Bz: np.ndarray,
    polarity_threshold: float = 200.0,
    complexity_index: float = 0.5,
    has_common_penumbra_delta: bool = False,
) -> str:
    """Rule-based Hale classifier for an AR.

    Args:
        Bz: 2-D LOS magnetogram [Gauss].
        polarity_threshold: |Bz| above which a polarity is counted [Gauss].
        complexity_index: Fractal-like index in [0,1]; > 0.5 indicates complex.
        has_common_penumbra_delta: True if any opposite-polarity umbrae share
            a common penumbra (<2° separation).

    Returns:
        One of 'alpha', 'beta', 'beta_gamma', 'beta_gamma_delta'.
    """
    pos_flux = Bz[Bz > polarity_threshold].sum()
    neg_flux = -Bz[Bz < -polarity_threshold].sum()
    total = pos_flux + neg_flux + 1e-9
    balance = min(pos_flux, neg_flux) / total

    if balance < 0.05:
        return "alpha"
    if has_common_penumbra_delta:
        return "beta_gamma_delta"
    if complexity_index > 0.5:
        return "beta_gamma"
    return "beta"


hale = classify_hale(Bz, complexity_index=0.6, has_common_penumbra_delta=True)
print(f"Synthetic AR Hale classification: {hale}")

from collections import Counter
results = []
for f, cls_true in zip(pop["flux"], pop["hale_class"]):
    complexity = min(1.0, (np.log10(f) - 21.0) / 2.3)
    has_delta = cls_true == "beta_gamma_delta"
    pred = classify_hale(
        np.array([[f / 1e22 * 1000, -f / 1e22 * 900]]),
        complexity_index=complexity,
        has_common_penumbra_delta=has_delta,
    )
    results.append((cls_true, pred))

confusion = Counter(results)
print("\nClassification agreement (true -> predicted: count):")
for (t, p), c in confusion.most_common():
    marker = "OK" if t == p else "!!"
    print(f"  [{marker}] {t:20s} -> {p:20s} : {c}")

## Part 6: Flare Forecasting Probability by AR Class / AR 분류에 따른 예보 확률

We combine the Hale class with the R-value (Schrijver 2007) and helicity-flux threshold (LaBonte 2007) to produce a simple multivariate probability estimate for X-class flare within 24 hours. This mirrors the logic of the SHARP+ML pipeline (Bobra & Couvidat 2015) in simplified form.

Hale 분류 + R-value(Schrijver 2007) + helicity flux(LaBonte 2007) 임계를 결합해 24시간 내 X-class 확률을 간단히 추정한다. SHARP+ML(Bobra & Couvidat 2015) 파이프라인의 단순화 형태다.

In [ ]:
def forecast_x_class_probability(
    hale_class: str,
    log_R: float,
    peak_helicity_flux: float,
    non_neutralized_current: float,
) -> float:
    """Estimate probability of X-class flare within 24 hours.

    Args:
        hale_class: One of 'alpha', 'beta', 'beta_gamma', 'beta_gamma_delta'.
        log_R: log10 Schrijver R-value in Mx.
        peak_helicity_flux: Peak dH/dt in Mx^2 s^-1.
        non_neutralized_current: Total |J| in A.

    Returns:
        Approximate probability in [0, 1].
    """
    base = {
        "alpha": 0.005,
        "beta": 0.02,
        "beta_gamma": 0.10,
        "beta_gamma_delta": 0.40,
    }[hale_class]

    r_bump = max(0.0, min(0.2, 0.2 * (log_R - 4.5) / 0.5))
    h_bump = 0.25 if peak_helicity_flux > 6e36 else 0.0
    j_bump = 0.15 if non_neutralized_current >= 4.6e12 else 0.0

    return min(0.95, base + r_bump + h_bump + j_bump)


scenarios = [
    ("Quiet α spot", "alpha", 3.0, 1e35, 1e11),
    ("Simple β bipole (RGO 14886-like)", "beta", 4.0, 5e35, 5e11),
    ("Complex βγ medium AR", "beta_gamma", 4.7, 2e36, 3e12),
    ("AR 10930 (X3.4 producer)", "beta_gamma_delta", 5.2, 7e36, 5e12),
    ("AR 12673 (X9.3 producer)", "beta_gamma_delta", 5.4, 1.5e37, 8e12),
]

print("Scenario                                 | P(X, 24h)")
print("-" * 60)
for label, hale, logR, dHdt, J in scenarios:
    p = forecast_x_class_probability(hale, logR, dHdt, J)
    print(f"{label:40s} | {p:.2f}")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| δ-sunspot morphology | Opposite polarities within common penumbra (Künzel 1960) | HMI vector magnetogram + penumbra segmentation |
| Free magnetic energy | ∫(B²_NLFFF − B²_Pot)/8π dV | SHARP TOTPOT parameter, NLFFF codes |
| Shear angle along PIL | Angle vs. potential reference; 80–90° for flaring PIL | SHARP MEANSHR, MEANGBT |
| Hale class decision tree | α → β → βγ → βγδ | Mt. Wilson categorical ML feature |
| Helicity injection threshold | dH/dt > 6×10³⁶ Mx² s⁻¹ (LaBonte 2007) | DAVE4VM flow-tracking + Poynting-flux |
| Multivariate flare forecast | SHARP F-scores (Bobra & Couvidat 2015) | Nishizuka et al. 2018 DeFN deep NN |

이 구현은 리뷰 논문의 관측 진단·분류·예보 논리를 단순 장난감 모델로 재현한다. 실제 예보 파이프라인은 SHARP 파라미터와 딥러닝 분류기(Bobra 2015; Nishizuka 2018)를 사용한다.

### References / 참고문헌
- Toriumi & Wang 2019, *Living Reviews in Solar Physics*, 16:3.
- Künzel 1960, *Astron. Nachr.* 285, 271 (δ-classification).
- Sammis et al. 2000, *ApJ* 540, 583 (flare vs. Mt. Wilson class).
- Schrijver 2007, *ApJ Lett.* 655, L117 (R-value).
- LaBonte et al. 2007, *ApJ* 671, 955 (helicity-flux threshold).
- Bobra & Couvidat 2015, *ApJ* 798, 135 (SHARP + ML).
- Nishizuka et al. 2018, *ApJ* 858, 113 (DeFN).
- Kontogiannis et al. 2017, *Solar Phys.* 292, 159 (non-neutralized current).